# Traditional StudyBot
Rule-based Python; no LLM or API. Run cells in order. Local execution evidence is embedded; Colab deployment has not been verified.

In [1]:
from pathlib import Path
Path("chatbot.py").write_text('"""Deterministic study assistant; standard library only, no model or API."""\nimport re\nimport unicodedata\nfrom dataclasses import dataclass\n\n@dataclass(frozen=True)\nclass Reply:\n    text: str\n    intent: str\n    end: bool = False\n\nclass StudyBot:\n    """One instance per session. respond is the future service integration seam."""\n    def __init__(self):\n        self.waiting_minutes = False\n\n    def respond(self, message):\n        if not isinstance(message, str):\n            return Reply(\'Please enter text, such as help.\', \'invalid\')\n        if len(message) > 500:\n            return Reply(\'Please shorten your message to 500 characters or fewer.\', \'invalid\')\n        text = \' \'.join(unicodedata.normalize(\'NFKC\', message).casefold().split())\n        text = text.strip(\' .!?\')\n        if not text or not any(c.isalnum() for c in text):\n            return Reply(\'Please type a question or enter help.\', \'invalid\')\n        if text in (\'bye\', \'quit\', \'exit\'):\n            self.waiting_minutes = False\n            return Reply(\'Goodbye. Good luck with your studies!\', \'goodbye\', True)\n        if text in (\'cancel\', \'reset\'):\n            self.waiting_minutes = False\n            return Reply(\'Study planning canceled. Enter help for topics.\', \'cancel\')\n        if text in (\'help\', \'menu\', \'capabilities\', \'what can you do\'):\n            return Reply(\'I can explain HCI, explain a chatbot lifecycle, give testing tips, and plan a study session. Try: HCI, lifecycle, testing, or study plan. Use cancel to reset or bye to exit.\', \'help\')\n        if self.waiting_minutes:\n            if not re.fullmatch(r\'[0-9]{1,3}\', text) or not 10 <= int(text) <= 180:\n                return Reply(\'Enter a whole number from 10 to 180 minutes, or cancel.\', \'invalid_minutes\')\n            minutes = int(text)\n            self.waiting_minutes = False\n            review = minutes // 3\n            return Reply(f\'For {minutes} minutes: review for {review}, practice for {minutes - review - 5}, and recap for 5.\', \'plan\')\n        if re.search(r\'\\b(study plan|plan a study session)\\b\', text):\n            self.waiting_minutes = True\n            return Reply(\'How many minutes do you have? Enter 10 to 180, or cancel.\', \'ask_minutes\')\n        topics = [\n            (\'hci\', r\'\\b(hci|human computer interaction|human-computer interaction)\\b\', \'HCI studies how people interact with computers. A clear interface gives feedback and helps users recover from errors.\'),\n            (\'lifecycle\', r\'\\b(lifecycle|life cycle)\\b\', \'A chatbot lifecycle includes requirements, conversation design, implementation, testing, deployment, and maintenance.\'),\n            (\'testing\', r\'\\b(testing|test|tests)\\b\', \'Test expected prompts, paraphrases, ambiguous requests, malformed input, and multi-turn state transitions.\')\n        ]\n        hits = [(name, answer) for name, pattern, answer in topics if re.search(pattern, text)]\n        if len(hits) > 1:\n            return Reply(\'Please ask about one topic at a time: HCI, lifecycle, or testing.\', \'clarify\')\n        if hits:\n            return Reply(hits[0][1], hits[0][0])\n        if text in (\'hi\', \'hello\', \'hey\'):\n            return Reply(\'Hello! I am StudyBot, a rule-based study assistant. Enter help to see my capabilities.\', \'greeting\')\n        return Reply(\'I do not recognize that request. Enter help for supported topics.\', \'fallback\')\n\ndef main():\n    bot = StudyBot()\n    print(\'StudyBot | Traditional rule-based chatbot | Enter help or bye\')\n    while True:\n        try:\n            message = input(\'You: \')\n        except (EOFError, KeyboardInterrupt):\n            print(\'\\nStudyBot: Goodbye!\')\n            break\n        reply = bot.respond(message)\n        print(\'StudyBot:\', reply.text)\n        if reply.end:\n            break\n\nif __name__ == \'__main__\':\n    main()\n', encoding="utf-8")
Path("test_chatbot.py").write_text("import unittest\nfrom chatbot import StudyBot\n\nclass StudyBotTests(unittest.TestCase):\n    def setUp(self):\n        self.bot = StudyBot()\n    def test_supported_prompts(self):\n        for text, intent in [('hi','greeting'),('help','help'),('What is HCI?','hci'),('Explain the life cycle','lifecycle'),('testing tips','testing'),('bye','goodbye')]:\n            with self.subTest(text=text): self.assertEqual(self.bot.respond(text).intent,intent)\n    def test_normalization(self):\n        self.assertEqual(self.bot.respond('  ＨＣＩ  !!! ').intent, 'hci')\n    def test_word_boundaries(self):\n        self.assertEqual(self.bot.respond('contest').intent,'fallback')\n    def test_ambiguous_topics(self):\n        self.assertEqual(self.bot.respond('HCI and testing').intent,'clarify')\n    def test_malformed(self):\n        for value in ['', '  ', '?!', None, 42, 'x'*501]:\n            with self.subTest(value=str(value)[:20]): self.assertEqual(self.bot.respond(value).intent,'invalid')\n    def test_unknown(self):\n        self.assertEqual(self.bot.respond('weather tomorrow').intent,'fallback')\n    def test_plan(self):\n        self.assertEqual(self.bot.respond('study plan').intent,'ask_minutes')\n        self.assertIn('review for 10, practice for 15, and recap for 5', self.bot.respond('30').text)\n        self.assertFalse(self.bot.waiting_minutes)\n    def test_bad_minutes_keep_state(self):\n        self.bot.respond('study plan')\n        for value in ['9','181','20.5','-10','ten']:\n            self.assertEqual(self.bot.respond(value).intent,'invalid_minutes')\n            self.assertTrue(self.bot.waiting_minutes)\n    def test_plan_boundaries(self):\n        for value in ['10','180']:\n            self.bot.respond('study plan')\n            self.assertEqual(self.bot.respond(value).intent,'plan')\n    def test_help_keeps_state(self):\n        self.bot.respond('study plan')\n        self.assertEqual(self.bot.respond('help').intent,'help')\n        self.assertTrue(self.bot.waiting_minutes)\n    def test_cancel(self):\n        self.bot.respond('study plan')\n        self.bot.respond('cancel')\n        self.assertEqual(self.bot.respond('HCI').intent,'hci')\n    def test_exit(self):\n        self.bot.respond('study plan')\n        self.assertTrue(self.bot.respond('bye').end)\n        self.assertFalse(self.bot.waiting_minutes)\n    def test_session_isolation(self):\n        self.bot.respond('study plan')\n        self.assertEqual(StudyBot().respond('30').intent,'fallback')\n    def test_determinism(self):\n        self.assertEqual(self.bot.respond('HCI'),self.bot.respond('HCI'))\n\nif __name__ == '__main__':\n    unittest.main(verbosity=2)\n", encoding="utf-8")
print("Created chatbot.py and test_chatbot.py")

Created chatbot.py and test_chatbot.py


In [2]:
from chatbot import StudyBot
bot = StudyBot()
for prompt in ['help', 'What is HCI?', 'study plan', 'ten', '30', '???', 'weather tomorrow', 'bye']:
    print("You:", prompt)
    print("StudyBot:", bot.respond(prompt).text)
    print()

You: help
StudyBot: I can explain HCI, explain a chatbot lifecycle, give testing tips, and plan a study session. Try: HCI, lifecycle, testing, or study plan. Use cancel to reset or bye to exit.

You: What is HCI?
StudyBot: HCI studies how people interact with computers. A clear interface gives feedback and helps users recover from errors.

You: study plan
StudyBot: How many minutes do you have? Enter 10 to 180, or cancel.

You: ten
StudyBot: Enter a whole number from 10 to 180 minutes, or cancel.

You: 30
StudyBot: For 30 minutes: review for 10, practice for 15, and recap for 5.

You: ???
StudyBot: Please type a question or enter help.

You: weather tomorrow
StudyBot: I do not recognize that request. Enter help for supported topics.

You: bye
StudyBot: Goodbye. Good luck with your studies!



In [3]:
import subprocess, sys
result = subprocess.run([sys.executable, "-m", "unittest", "-v"], capture_output=True, text=True)
print(result.stdout + result.stderr)
assert result.returncode == 0

test_ambiguous_topics (test_chatbot.StudyBotTests.test_ambiguous_topics) ... ok
test_bad_minutes_keep_state (test_chatbot.StudyBotTests.test_bad_minutes_keep_state) ... ok
test_cancel (test_chatbot.StudyBotTests.test_cancel) ... ok
test_determinism (test_chatbot.StudyBotTests.test_determinism) ... ok
test_exit (test_chatbot.StudyBotTests.test_exit) ... ok
test_help_keeps_state (test_chatbot.StudyBotTests.test_help_keeps_state) ... ok
test_malformed (test_chatbot.StudyBotTests.test_malformed) ... ok
test_normalization (test_chatbot.StudyBotTests.test_normalization) ... ok
test_plan (test_chatbot.StudyBotTests.test_plan) ... ok
test_plan_boundaries (test_chatbot.StudyBotTests.test_plan_boundaries) ... ok
test_session_isolation (test_chatbot.StudyBotTests.test_session_isolation) ... ok
test_supported_prompts (test_chatbot.StudyBotTests.test_supported_prompts) ... ok
test_unknown (test_chatbot.StudyBotTests.test_unknown) ... ok
test_word_boundaries (test_chatbot.StudyBotTests.test_word_bou

## Optional interactive conversation
Uncomment and run the following line to chat. Enter bye to stop.

In [5]:
# from chatbot import main; main()